# 01 · Data Cleaning — Tracy Temple Feasibility (Colab)

**All data in this project is synthetic** (see README). This notebook is your workspace for Part 2.

How to use it:
- The setup, loading, inspection, logging and export cells are done for you.
- Cells marked **TODO** are yours. Each one references a rule in `docs/04_data_quality_challenge_guide.md`.
- After each judgment call, add a short markdown note: *what I decided, why, what the alternative was.*

## 0 · Get the project files
**Option A (recommended):** your GitHub repo is public → clone it.  
**Option B:** upload the project zip.

In [ ]:
# Option A - clone your repo
REPO_URL = "https://github.com/YOUR-USERNAME/tracy-temple-feasibility.git"   # <- change this
!git clone -q {REPO_URL}
%cd tracy-temple-feasibility

In [ ]:
# Option B - upload the zip instead (skip if you used Option A)
# from google.colab import files
# up = files.upload()                       # choose tracy-temple-feasibility-part2.zip
# !unzip -q -o tracy-temple-feasibility-part2.zip
# %cd tracy-temple-feasibility

## 1 · Load raw data
Everything is read **as text** (`dtype=str, keep_default_na=False`). That's deliberate: if pandas guesses types on load, it silently turns `"N/A"` into NaN and `"$1,200"` into a string column, and you lose the evidence of what the raw value was. You decide the types.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

RAW, REF, CLEAN = Path("data/raw"), Path("data/reference"), Path("data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50, "display.width", 200)

def read_raw(name):
    return pd.read_csv(RAW / name, dtype=str, keep_default_na=False)

raw = {f.stem: read_raw(f.name) for f in sorted(RAW.glob("*.csv"))}
ref = {f.stem: pd.read_csv(REF / f.name) for f in sorted(REF.glob("*.csv"))}

for name, df in raw.items():
    print(f"{name:30s} {df.shape[0]:5d} rows x {df.shape[1]:2d} cols")

## 2 · Inspect
Look before you clean. For every categorical column, list the distinct raw values and how often they appear.

In [ ]:
def profile(df, max_unique=40):
    """Distinct values per column (with counts) for columns with few unique values."""
    for col in df.columns:
        vc = df[col].value_counts(dropna=False)
        if len(vc) <= max_unique:
            print(f"\n--- {col} ({len(vc)} distinct)")
            print(vc.to_string())
        else:
            blanks = (df[col].str.strip() == "").sum()
            print(f"\n--- {col}: {len(vc)} distinct, {blanks} blank; sample: {list(vc.index[:5])}")

survey_raw = raw["community_survey"]
survey_raw.head()

In [ ]:
profile(survey_raw)

In [ ]:
# TODO: profile the other raw tables too (raw["program_preferences"], raw["candidate_properties"], ...)

## 3 · Issue log
Every change you make gets logged here. This becomes `dq_issue_log.csv` and is loaded into the database, which is your audit trail.

In [ ]:
dq_log = []

def log_issue(table, key, column, rule_id, original, cleaned, action):
    """action must be one of: corrected, standardized, set_null, flagged, excluded_from_metric"""
    assert action in {"corrected", "standardized", "set_null", "flagged", "excluded_from_metric"}
    dq_log.append(dict(table_name=table, record_key=key, column_name=column, rule_id=rule_id,
                       original_value=None if original is None else str(original),
                       cleaned_value=None if cleaned is None else str(cleaned),
                       action_taken=action))

def log_changes(table, keys, column, rule_id, before, after, action="standardized"):
    """Log every row where a vectorised cleaning step changed the value."""
    changed = before.fillna("<NA>").astype(str) != after.fillna("<NA>").astype(str)
    for k, b, a in zip(keys[changed], before[changed], after[changed]):
        log_issue(table, k, column, rule_id, b, None if pd.isna(a) else a, action)
    return int(changed.sum())

## 4 · Warm-up: small tables
Start here: startup/operating costs, program options, funding scenarios. (Rules DQ-29 to DQ-34.)

In [ ]:
# TODO DQ-05: write a reusable function that turns '$1,200', '1200.00', '$95/hr' into a float, and unknown tokens into NaN
def to_number(x):
    ...

In [ ]:
# TODO DQ-29 to DQ-32: clean raw["startup_cost_estimates"] -> startup_cost_estimate
# Hint: raw column names differ from the schema (operating_model -> model_code, cost_category -> cost_category_code)

In [ ]:
# TODO: clean raw["operating_cost_estimates"], raw["program_options"], raw["funding_scenarios"]

## 5 · Properties & due diligence (DQ-22 to DQ-28)

In [ ]:
# TODO: clean raw["candidate_properties"] -> candidate_property

In [ ]:
# TODO: clean raw["property_due_diligence"] -> property_due_diligence
# Check at the end: every property_id has exactly 5 rows and dd_id is unique

## 6 · Survey (DQ-03, DQ-07 to DQ-16)
The hardest part. Suggested order: dates → categoricals → numerics → explode lists → duplicates → flags.

In [ ]:
# TODO DQ-03: parse response_date. Count how many rows used each format, and how many failed.

In [ ]:
# TODO DQ-07: mapping dictionaries for attendance_frequency, preferred_day, preferred_time, response_channel, age_group, household_size_range

In [ ]:
# TODO DQ-13: explode preferred_programs -> survey_top_choice (respondent_id, category_code, choice_rank)
alias = dict(zip(ref["program_category_alias"].alias_text, ref["program_category_alias"].category_code))

In [ ]:
# TODO DQ-14: explode main_barriers -> survey_barrier (respondent_id, barrier_code)

In [ ]:
# TODO DQ-10: build a fingerprint per BRL-02 on the CLEANED values and flag duplicates within 48 hours
# Checkpoint: 13 flagged duplicates

## 7 · Preferences (DQ-17 to DQ-21)

In [ ]:
# TODO: clean raw["program_preferences"] -> program_preference

## 8 · Validate before export
Re-check primary keys, foreign keys and allowed values in pandas **before** you try PostgreSQL.

In [ ]:
def check_pk(df, cols, name):
    d = df.duplicated(cols, keep=False).sum()
    print(f"{name}: {d} rows with duplicate key {cols}")

def check_fk(child, col, parent, pcol, name):
    orphans = ~child[col].isin(parent[pcol])
    print(f"{name}: {orphans.sum()} orphan {col} values")

# TODO: run check_pk / check_fk on every clean table, e.g.
# check_pk(survey_response, ["respondent_id"], "survey_response")
# check_fk(program_preference, "respondent_id", survey_response, "respondent_id", "program_preference")

## 9 · Before vs after (FR-15)
Fill in the five metrics from the challenge guide for raw vs clean, then write 2-3 sentences on what changed most.

In [ ]:
# TODO: build a DataFrame with columns metric | raw | clean | change

## 10 · Export in schema column order

In [ ]:
SCHEMA_COLUMNS = {
    "survey_response": ["respondent_id","response_date","response_channel","consent_flag","age_group",
        "household_size_range","attendance_frequency","preferred_day","preferred_time","max_travel_minutes",
        "monthly_contribution_stated","one_time_pledge_stated","volunteer_hours_month","comment_text",
        "is_duplicate","duplicate_of","dq_flag_count","data_status","source_id"],
    "survey_top_choice": ["respondent_id","category_code","choice_rank"],
    "survey_barrier": ["respondent_id","barrier_code"],
    "program_preference": ["preference_id","respondent_id","category_code","interest_rating","preferred_frequency",
        "data_status","source_id"],
    "program_option": ["program_id","category_code","program_name","intended_audience","est_launch_cost",
        "est_monthly_cost","volunteer_hours_month","paid_hours_month","min_area_sqft","requires_open_flame",
        "requires_kitchen","requires_permanent_space","evidence_note","verification_status","data_status","source_id"],
    "candidate_property": ["property_id","area_label","jurisdiction","property_type","tenure_offered",
        "floor_area_sqft","lot_area_acres","hourly_rate","asking_monthly_rent","asking_price","parking_spaces",
        "current_use","building_condition","zoning_label","zoning_verification","date_verified","is_simulated",
        "data_status","source_id"],
    "property_due_diligence": ["dd_id","property_id","review_area","status","est_cost_to_resolve","next_action",
        "owner_role","last_updated","data_status","source_id"],
    "startup_cost_estimate": ["cost_id","model_code","cost_category_code","line_description","low_estimate",
        "base_estimate","high_estimate","estimate_basis","verification_status","data_status","source_id"],
    "operating_cost_estimate": ["op_cost_id","model_code","cost_category_code","line_description","monthly_low",
        "monthly_base","monthly_high","estimate_basis","verification_status","data_status","source_id"],
    "funding_scenario": ["scenario_id","scenario_name","initial_funds_assumed","monthly_recurring_assumed",
        "ramp_months","pledge_realization_rate","annual_event_fundraising_net","grant_amount_assumed",
        "attendance_discount","confidence_level","assumptions_text","data_status","source_id"],
    "model_assumption": ["assumption_id","assumption_name","value_low","value_base","value_high","unit",
        "applies_to","rationale","verification_status","data_status","source_id"],
}

def export(name, df):
    """Write a clean table with schema column order. Integer columns with NaN should be pandas 'Int64'
    so they don't export as 12.0."""
    out = df[SCHEMA_COLUMNS[name]]
    out.to_csv(CLEAN / f"{name}.csv", index=False)
    print(f"wrote {name}.csv  ({len(out)} rows)")

# TODO: export(...) each clean table. Remember: survey_response sorted with is_duplicate == False first.
# pd.DataFrame(dq_log).to_csv(CLEAN / "dq_issue_log.csv", index=False)

In [ ]:
# Download everything as one zip
# !cd data && zip -qr clean.zip clean
# from google.colab import files; files.download("data/clean.zip")